# 03 · Senses: how a Mario object becomes a fly's input (looming)

**Goal:** understand what a **sense** is, by building the inputs of the *looming* sense by hand and watching how it behaves.

You will learn
- the sense contract (`contracts.py`): `cue(objects, mario, facts, knobs, memory) -> SenseOutput`
- what looming computes: the **angle** an object would subtend in a fly's eye, and its **change**
- how the nearest object is chosen, how the shapes (free / wall / tower) differ, how noise and knobs act
- the **honesty rule**: which parts are real fly (teal) and which are invented by us (orange)

No MaleCNS data needed. We build tiny hand-made scenes, the same way `tests/test_senses_behaviour.py` does.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import _helpers as H
from fly_mario_fly import registry
from fly_mario_fly.contracts import Box, MarioState, ObjectInstance
from fly_mario_fly.mario.facts import load_facts
from fly_mario_fly.senses.kit.geom import theta_of
from fly_mario_fly.senses.kit.memory import SenseMemory

## 1. Meet the sense

Senses are plug-ins in `src/fly_mario_fly/senses/` (one file each, one object called `PLUGIN`). Let's list them and open the looming one.

In [ ]:
senses = registry.list_plugins("senses")
for name, s in senses.items():
    print(f"{name:12s} scope={s.scope:10s} feeds={s.feeds}   {s.label}")

In [ ]:
looming = senses["looming"]
print("label  :", looming.label)
print("effect :", looming.effect)
print("scope  :", looming.scope, " (per_object = it is attached to object types; whole_fly = it looks at everything)")
print("feeds  :", looming.feeds, " (which fly circuit it drives: see fly/mechanisms/escape.py)")
print()
for k in looming.knobs:
    print(f"knob {k.key:15s} default={k.default}  range=[{k.lo}, {k.hi}]  origin={k.origin}")

A **sense** is a bridge: game objects in, **neuron input** out. It outputs numbers for *neuron groups* (`LC4_R`, `LPLC2_R`, ... defined in `fly/groups.yaml`).
The sense knows nothing about the brain's wiring; the brain knows nothing about Mario. Only `senses/`, `actions/` and `runner/` may know both.

## 2. The idea: angle, not distance

A fly sees a world where an approaching object **grows** on its retina. The NES view is side-on, so nothing really expands on screen. The project's rule (written at the top of `senses/looming.py`)
is to *synthesise* the angle from RAM, as seen from Mario's eye:

$$\theta = 2\,\arctan\!\left(\frac{h}{2d}\right)$$

with *h* the object's height and *d* the true 2-D gap between Mario's box and the object's box. Let's plot it.

In [ ]:
d = np.linspace(4, 200, 200)
fig, ax = plt.subplots(figsize=(7, 3.6))
for h, label in [(16, "goomba / block (16 px)"), (24, "koopa (24 px)"), (32, "short pipe (32 px)")]:
    ax.plot(d, [theta_of(x, h) for x in d], label=label)
ax.set_xlabel("gap d (px)"); ax.set_ylabel("theta (radians)"); ax.legend(); ax.set_title("looming angle grows fast when something is close")
plt.show()

`theta_of` is the repo's own helper (`senses/kit/geom.py`). Small far-away things barely register; the angle explodes in the last ~30 px. A looming detector cell reacts to *that*.

## 3. Call the sense by hand

`cue(objects, mario, facts, knobs, memory)` needs five things:

| argument | what | here |
|---|---|---|
| `objects` | the `ObjectInstance`s this sense is attached to | a hand-made goomba |
| `mario` | `MarioState` | standing at x = 100 |
| `facts` | the game facts (notebook 06) | `load_facts("1-1")` |
| `knobs` | dial values | the defaults |
| `memory` | per-sense memory kept between frames | `SenseMemory()` |

In [ ]:
facts = load_facts("1-1")
KNOBS = {k.key: k.default for k in looming.knobs}

def mario_at(x=100.0, feet=176.0, h=16.0, airborne=False):
    return MarioState(x=x, y=feet - h, w=12.0, h=h, airborne=airborne, power="small", facing=1)

def goomba(slot, x, top=160.0, h=16.0):
    return ObjectInstance("goomba", slot, Box(x, top, 16, h), "ahead", {"slot": slot, "shell": False})

def new_memory(world=(), sigma=0.0, seed=0):
    mem = SenseMemory()
    mem.rng, mem.sigma, mem.world, mem.mods = np.random.default_rng(seed), sigma, list(world), {}
    return mem

def cue(objects, mario, mem, knobs=None, world=None):
    mem.world = list(objects if world is None else world)
    return looming.cue(list(objects), mario, facts, {**KNOBS, **(knobs or {})}, mem)

out = cue([goomba(0, 160)], mario_at(), new_memory())
print("inputs  :", out.inputs)
print()
for step in out.steps:
    print(f"{step.label:10s} {step.formula:35s} = {step.value:.3f}")

- `inputs` is what goes into the brain: `LPLC2_R` gets θ. `LC4_R` is 0 because this is a *new* target: there's no change to report yet.
- `steps` is the sense's worked calculation. The web app shows these lines to the user, so every number is explainable.
- `out.overlay` and `out.modifiers` are for drawing and for senses/actions that read each other (we'll ignore them).

## 4. An approaching goomba, frame by frame

The memory carries the previous θ from frame to frame, so LC4 (which answers *motion*, not size) can report |change of θ|.

In [ ]:
mem = new_memory()
mario = mario_at(100)
frames, theta, lc4, gap = [], [], [], []
x = 260.0                                   # goomba starts far to the right and walks left 1 px per frame
for f in range(200):
    out = cue([goomba(0, x)], mario, mem)
    frames.append(f); theta.append(out.inputs["LPLC2_R"]); lc4.append(out.inputs["LC4_R"]); gap.append(x - 112)
    x -= 1
fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
axes[0].plot(frames, theta, color="tab:green"); axes[0].set_ylabel("LPLC2_R input (theta)")
axes[1].plot(frames, lc4, color="tab:blue"); axes[1].set_ylabel("LC4_R input (|change|)"); axes[1].set_xlabel("frame")
axes[0].set_title("a goomba walking toward a standing Mario")
plt.show()

Two different channels from the same object: **size** (LPLC2) rises smoothly; **rate of growth** (LC4) is tiny until the very end, then jumps. Real LC4 and LPLC2 cells (both are looming detectors in the fly's visual system) are
tuned differently, which is why the repo feeds them different things.

## 5. Which object? The nearest threat only

Only **one** object feeds the detectors at a time: the nearest by gap, re-chosen every frame. When the target switches, the change is reported as 0 (a new object didn't "grow").

In [ ]:
mem = new_memory()
a = cue([goomba(0, 200)], mario, mem)
b = cue([goomba(0, 199)], mario, mem)                       # same goomba, 1 px closer
c = cue([goomba(0, 198), goomba(1, 135)], mario, mem)       # a second goomba appears much nearer
print("1) one goomba      : theta =", round(a.inputs["LPLC2_R"], 4), " change =", round(a.inputs["LC4_R"], 5))
print("2) 1 px closer     : theta =", round(b.inputs["LPLC2_R"], 4), " change =", round(b.inputs["LC4_R"], 5))
print("3) nearer one shows: theta =", round(c.inputs["LPLC2_R"], 4), " change =", round(c.inputs["LC4_R"], 5), " <- target switched, change is 0")
print("   which gap was used? ", [s.value for s in c.steps if s.label == "distance"])

## 6. Shapes: how an object *looms*

Not everything is a little box. In `senses/shapes/` each shape is a plug-in:

| shape | used for | idea |
|---|---|---|
| `own_box` | goombas, koopas, pickups | the object's own box |
| `solid_block` | pipes, steps | a **wall** standing on the floor: looms from the ground Mario took off from up to its top |
| `far_edge_tower` | pits | the tower on a pit's far edge looms like a wall; the hole itself is the base |
| `custom_box` | anything you want | your own offset and scale |

Which shape an object type uses comes from its plug-in (`default_loom_shape`) and can be overridden in a fly's config (`loom_shapes:`).

In [ ]:
print({name: p.default_loom_shape for name, p in registry.list_plugins("objects").items()})
print(list(registry.list_plugins("shapes")))

In [ ]:
def pipe(x, height=32):
    return ObjectInstance("pipe", x, Box(x, 176 - height, 32, height), "ahead")

def pit(x, w=32):
    return ObjectInstance("pit", x, Box(x, 176, w, 32), "ahead", {"open": False})

gaps = np.arange(10, 150, 5)
res = {"goomba (own_box, 16 px)": [], "pipe (wall, 32 px)": [], "pit (tower)": []}
for g in gaps:
    x = 112 + g
    res["goomba (own_box, 16 px)"].append(cue([goomba(0, x)], mario, new_memory()).inputs["LPLC2_R"])
    res["pipe (wall, 32 px)"].append(cue([pipe(x)], mario, new_memory()).inputs["LPLC2_R"])
    res["pit (tower)"].append(cue([pit(x)], mario, new_memory()).inputs["LPLC2_R"])
fig, ax = plt.subplots(figsize=(7, 3.6))
for k, v in res.items():
    ax.plot(gaps, v, label=k)
ax.set_xlabel("gap between Mario and the object (px)"); ax.set_ylabel("LPLC2_R input"); ax.legend()
ax.set_title("the same Mario, three kinds of thing")
plt.show()

A tall wall looms more than a goomba at the same gap, as an eye would see it. A pit behaves differently because of its hole-plus-tower shape (see the long docstring at the top of `senses/looming.py`).

## 7. Knobs, and who made them up

`loom.eyes` switches on the left eye as well. A **knob** (`KnobSpec` in `contracts.py`) is a user-visible, evolvable number with a range and an **origin**:
`"fly"` = read off the real connectome, `"invented"` = our choice.

In [ ]:
right_only = cue([goomba(0, 160)], mario, new_memory())
both = cue([goomba(0, 160)], mario, new_memory(), knobs={"loom.eyes": 1.0})
print("loom.eyes = 0:", sorted(right_only.inputs))
print("loom.eyes = 1:", sorted(both.inputs))

### The honesty rule: teal = real wiring, orange = invented by us

For *this* sense, be careful what is real:
- 🟦 **teal (real):** what LC4 and LPLC2 connect to — they really do drive the Giant Fiber in the MaleCNS data (notebooks 01, 02, 04).
- 🟧 **orange (invented):** the **cue** itself: θ = 2·atan(h/2d) computed from NES RAM, the choice of "nearest object only", the gain between θ and neuron input, the shapes. The fly never saw a NES.

A good fly result says "this invented cue, driving this real circuit, did X". The web app colours every dial teal or orange accordingly.

In [ ]:
for name, s in senses.items():
    origins = sorted({k.origin for k in s.knobs}) or ["(no knobs)"]
    print(f"{name:12s} knob origins: {origins}")

## 8. Noise

`sigma` is *sensor noise*: a fraction of each input, drawn from a seeded random generator (`np.random.default_rng(seed)`). Same seed → same noise → the same run. That is how the repo
tests robustness while staying reproducible.

In [ ]:
clean = [cue([goomba(0, 160)], mario, new_memory(sigma=0.0)).inputs["LPLC2_R"] for _ in range(3)]
mem = new_memory(sigma=0.05, seed=1)
noisy = [cue([goomba(0, 160)], mario, mem).inputs["LPLC2_R"] for _ in range(5)]
again = new_memory(sigma=0.05, seed=1)
noisy2 = [cue([goomba(0, 160)], mario, again).inputs["LPLC2_R"] for _ in range(5)]
print("no noise  :", np.round(clean, 4))
print("sigma 0.05:", np.round(noisy, 4))
print("same seed :", np.round(noisy2, 4), " <- identical")

## 9. On the real game

Now feed the sense *real* objects found in RAM (notebook 06) and draw the overlay it returns. The `OverlayItem`s are what the web app paints on top of the game.

In [ ]:
from fly_mario_fly.mario import ram as R
from fly_mario_fly.mario.env import MarioEnv

env = MarioEnv(); env.reset("1-1", noop=3)
objects = registry.list_plugins("objects")
for _ in range(400):
    env.step(["right"])
    if [o for o in objects["goomba"].find(env.ram, env.facts) if o.box.x - R.mario_x(env.ram, env.facts) < 70]:
        break
f = env.facts
mario_now = R.read_mario(env.ram, f)
gob = objects["goomba"].find(env.ram, f)
mem = new_memory(world=gob)
out = looming.cue(gob, mario_now, f, KNOBS, mem)
print("real goomba :", [o.box for o in gob])
print("inputs      :", out.inputs)
sx = int(env.ram[f["ram"]["screen_page"]]) * 256 + int(env.ram[f["ram"]["screen_x"]])
frame = env.step([]).frame
H.show_frame(frame, boxes=[o.geometry for o in out.overlay if o.kind == "box"], scroll=sx, title="the box the looming sense is looking at")
plt.show()

## Recap

- A sense is `cue(objects, mario, facts, knobs, memory) -> SenseOutput(inputs, overlay, modifiers, steps)`.
- Looming turns the **nearest** threat into θ (→ LPLC2) and |Δθ| (→ LC4). Shapes decide what counts as "the object".
- Everything the sense does is explained in `steps`; the noise is seeded; the origin of each dial is stated.
- The sense decides *what the fly receives*; it never decides what the fly does.

**Try this**
1. In section 4, make Mario jump: set `mario_at(100, feet=140, airborne=True)` and watch the shapes behave differently for a pit.
2. Use `loom_shapes` thinking: call the sense with `ObjectInstance("pipe", ...)` and a `custom_box` shape through the `loom_shapes` mechanism in `configs/flies/*.yaml`.
3. Open `senses/sweet_smell.py` and compare it with `looming.py`. What do they have in common?

**Next:** notebook 04 shows what the brain does with this input. (If you want the data first: notebooks 01 and 02.)